# Stage 04c -- Model Building (Forward Stepwise)

**Purpose:** Build a logistic regression PD model using forward stepwise variable selection from the approved variable shortlist.

**Inputs:**
- Binned dataset: `{RUN_DIR}/data/loans_binned.csv`
- Variable shortlist from Stage 03: 8 variables
- Target variable: `Creditability` (1 = default)

**Method:** Forward stepwise selection adds variables one at a time, choosing the variable that produces the most significant improvement (lowest p-value) at each step. Selection stops when no remaining variable has p-value below threshold.

**Note:** Since `Creditability = 1` denotes default (bad outcome) and WoE is defined as `ln(dist_good / dist_bad)`, WoE coefficients in the logistic regression predicting default should be negative. The `pdt.step_fwd()` function enforces positive WoE coefficients, which conflicts with this target encoding. Therefore, forward stepwise is implemented directly using statsmodels, accepting negative coefficients as the correct sign for WoE features predicting default.

In [ ]:
import sys
sys.path.insert(0, 'C:/projects/superagent/src')
import pdtoolkit as pdt
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import roc_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')

RUN_DIR = 'C:/projects/superagent/runs/2026-03-17_071354'

BLUE = '#2166AC'
RED = '#D6604D'
GREY = '#999999'
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 150

In [ ]:
# Load data
db_binned = pd.read_csv(f'{RUN_DIR}/data/loans_binned.csv')
print(f'Binned dataset shape: {db_binned.shape}')

shortlist = [
    'Account Balance',
    'Payment Status of Previous Credit',
    'Duration of Credit (month)',
    'Value Savings/Stocks',
    'Purpose',
    'Credit Amount',
    'Most valuable available asset',
    'Age (years)'
]

target = 'Creditability'

db_subset = db_binned[shortlist + [target]].copy()
print(f'Subset shape: {db_subset.shape}')
print(f'Target distribution:\n{db_subset[target].value_counts()}')

In [ ]:
# WoE-encode the binned variables
for col in shortlist:
    db_subset[col] = db_subset[col].astype(str)

db_woe, woe_info = pdt.replace_woe(db_subset, target)
print(f'WoE-encoded dataset shape: {db_woe.shape}')
if len(woe_info) > 0:
    print(f'WoE info issues:\n{woe_info}')
else:
    print('WoE encoding: No issues.')
print(f'\nSample:\n{db_woe.head()}')

In [ ]:
# Capture WoE mappings for each variable
woe_mappings = {}
for col in shortlist:
    woe_tbl_df = pdt.woe_tbl(db_subset, x=col, y=target)
    mapping_list = []
    for _, row in woe_tbl_df.iterrows():
        mapping_list.append({'bin': str(row['bin']), 'woe': float(row['woe'])})
    woe_mappings[col] = mapping_list
    print(f'{col}:')
    print(woe_tbl_df[['bin', 'woe']].to_string(index=False))
    print()

## Forward Stepwise Variable Selection

Custom forward stepwise implementation using p-value threshold = 0.05. At each step, the variable with the lowest p-value (most significant LR test) is added if below threshold.

WoE coefficients predicting `Creditability = 1` (default) are expected to be negative, since higher WoE indicates lower default risk.

In [ ]:
# Forward stepwise selection (manual implementation)
p_threshold = 0.05
y = db_woe[target].values

selected_variables = []
remaining = shortlist.copy()
step_records = []
iteration = 0

while remaining:
    iteration += 1
    candidates = []
    
    for rf in remaining:
        current_rfs = selected_variables + [rf]
        X_cand = sm.add_constant(db_woe[current_rfs].values)
        try:
            res = sm.Logit(y, X_cand).fit(disp=0, method='bfgs', maxiter=100)
            p_val = res.pvalues[-1]  # p-value for the new variable
            coef = res.params[-1]
            # For WoE predicting default (target=1), coefficient should be negative
            sign_ok = coef < 0
            # Compute AUC for this candidate model
            y_pred_cand = res.predict(X_cand)
            auc_cand = pdt.auc_model(y_pred_cand, y)
            candidates.append({
                'rf': rf, 'p_value': p_val, 'coefficient': coef,
                'sign_ok': sign_ok, 'aic': res.aic, 'auc': auc_cand
            })
        except Exception as e:
            print(f'  Error evaluating {rf}: {e}')
    
    if not candidates:
        break
    
    # Filter: p-value below threshold and correct sign
    valid = [c for c in candidates if c['p_value'] < p_threshold and c['sign_ok']]
    
    if not valid:
        # If no valid candidate with correct sign, try any with correct p-value
        # (coefficient sign will be checked later in self-assessment)
        valid = [c for c in candidates if c['p_value'] < p_threshold]
        if not valid:
            print(f'  Step {iteration}: No valid candidates (all p-values >= {p_threshold}). Stopping.')
            break
    
    # Select by lowest p-value
    best = min(valid, key=lambda x: x['p_value'])
    selected_variables.append(best['rf'])
    remaining.remove(best['rf'])
    
    step_records.append({
        'step': iteration,
        'variable_added': best['rf'],
        'lr_p_value': best['p_value'],
        'coefficient': best['coefficient'],
        'model_auc': best['auc'],
        'aic': best['aic']
    })
    print(f'  Step {iteration}: Added {best["rf"]} (p={best["p_value"]:.6f}, coef={best["coefficient"]:.4f}, AUC={best["auc"]:.4f})')

steps_df = pd.DataFrame(step_records)
print(f'\nSelected {len(selected_variables)} variables:')
for v in selected_variables:
    print(f'  - {v}')
print(f'\nStepwise steps:')
print(steps_df.to_string(index=False))

In [ ]:
# Self-assessment check 1: Variable count
p_threshold_used = p_threshold
threshold_adjusted = False
excluded_from_shortlist = [v for v in shortlist if v not in selected_variables]

if len(selected_variables) < 4:
    print(f'WARNING: Only {len(selected_variables)} variables selected. Relaxing p-value to 0.10.')
    p_threshold_used = 0.10
    threshold_adjusted = True
    
    # Re-run forward stepwise with relaxed threshold
    selected_variables = []
    remaining = shortlist.copy()
    step_records = []
    iteration = 0
    
    while remaining:
        iteration += 1
        candidates = []
        for rf in remaining:
            current_rfs = selected_variables + [rf]
            X_cand = sm.add_constant(db_woe[current_rfs].values)
            try:
                res = sm.Logit(y, X_cand).fit(disp=0, method='bfgs', maxiter=100)
                p_val = res.pvalues[-1]
                coef = res.params[-1]
                sign_ok = coef < 0
                y_pred_cand = res.predict(X_cand)
                auc_cand = pdt.auc_model(y_pred_cand, y)
                candidates.append({'rf': rf, 'p_value': p_val, 'coefficient': coef,
                                   'sign_ok': sign_ok, 'aic': res.aic, 'auc': auc_cand})
            except Exception:
                pass
        if not candidates:
            break
        valid = [c for c in candidates if c['p_value'] < p_threshold_used and c['sign_ok']]
        if not valid:
            valid = [c for c in candidates if c['p_value'] < p_threshold_used]
            if not valid:
                break
        best = min(valid, key=lambda x: x['p_value'])
        selected_variables.append(best['rf'])
        remaining.remove(best['rf'])
        step_records.append({'step': iteration, 'variable_added': best['rf'],
                             'lr_p_value': best['p_value'], 'coefficient': best['coefficient'],
                             'model_auc': best['auc'], 'aic': best['aic']})
    
    steps_df = pd.DataFrame(step_records)
    excluded_from_shortlist = [v for v in shortlist if v not in selected_variables]
    print(f'After relaxation: {len(selected_variables)} variables selected.')
    print(steps_df.to_string(index=False))

elif len(selected_variables) > 12:
    print(f'WARNING: {len(selected_variables)} variables selected. Tightening p-value to 0.01.')
    p_threshold_used = 0.01
    threshold_adjusted = True
    # Similar re-run with tighter threshold...
    selected_variables = []
    remaining = shortlist.copy()
    step_records = []
    iteration = 0
    while remaining:
        iteration += 1
        candidates = []
        for rf in remaining:
            current_rfs = selected_variables + [rf]
            X_cand = sm.add_constant(db_woe[current_rfs].values)
            try:
                res = sm.Logit(y, X_cand).fit(disp=0, method='bfgs', maxiter=100)
                p_val = res.pvalues[-1]
                coef = res.params[-1]
                sign_ok = coef < 0
                y_pred_cand = res.predict(X_cand)
                auc_cand = pdt.auc_model(y_pred_cand, y)
                candidates.append({'rf': rf, 'p_value': p_val, 'coefficient': coef,
                                   'sign_ok': sign_ok, 'aic': res.aic, 'auc': auc_cand})
            except Exception:
                pass
        if not candidates:
            break
        valid = [c for c in candidates if c['p_value'] < p_threshold_used and c['sign_ok']]
        if not valid:
            valid = [c for c in candidates if c['p_value'] < p_threshold_used]
            if not valid:
                break
        best = min(valid, key=lambda x: x['p_value'])
        selected_variables.append(best['rf'])
        remaining.remove(best['rf'])
        step_records.append({'step': iteration, 'variable_added': best['rf'],
                             'lr_p_value': best['p_value'], 'coefficient': best['coefficient'],
                             'model_auc': best['auc'], 'aic': best['aic']})
    steps_df = pd.DataFrame(step_records)
    excluded_from_shortlist = [v for v in shortlist if v not in selected_variables]
    print(f'After tightening: {len(selected_variables)} variables selected.')
else:
    print(f'Variable count check: PASS ({len(selected_variables)} variables)')

print(f'Excluded from shortlist: {excluded_from_shortlist}')

## Final Model Fit (Statsmodels)

In [ ]:
# Refit final model with statsmodels
X_woe = db_woe[selected_variables].copy()
X = sm.add_constant(X_woe)
logit_model = sm.Logit(y, X)
logit_res = logit_model.fit(disp=0)

print(logit_res.summary())

summary_df = pd.DataFrame({
    'variable': ['const'] + selected_variables,
    'coef': logit_res.params.values,
    'std_err': logit_res.bse.values,
    'z': logit_res.tvalues.values,
    'p_value': logit_res.pvalues.values,
    'ci_lower': logit_res.conf_int()[0].values,
    'ci_upper': logit_res.conf_int()[1].values
})
print('\nCoefficients table:')
print(summary_df.to_string(index=False))

In [ ]:
# Model-level statistics
model_stats = {
    'dep_variable': target,
    'n_obs': int(logit_res.nobs),
    'df_model': int(logit_res.df_model),
    'pseudo_r2': round(logit_res.prsquared, 4),
    'log_likelihood': round(logit_res.llf, 3),
    'll_null': round(logit_res.llnull, 3),
    'llr_pvalue': logit_res.llr_pvalue,
    'converged': bool(logit_res.mle_retvals['converged'])
}
for k, v in model_stats.items():
    print(f'  {k}: {v}')

## Self-Assessment Checks

In [ ]:
# Check 2: Coefficient sign consistency
# For WoE predicting default (target=1), coefficients should be NEGATIVE
# (higher WoE = lower risk = lower P(default))
signs_consistent = True
sign_issues = []
for var in selected_variables:
    coef = logit_res.params[var]
    if coef > 0:  # positive coefficient on WoE means sign reversal when predicting default
        signs_consistent = False
        sign_issues.append(var)
        print(f'  WARNING: {var} has positive coefficient ({coef:.4f}) -- sign reversal when predicting default!')

if signs_consistent:
    print('Coefficient sign check: PASS -- all WoE coefficients negative (correct for predicting default)')
else:
    print(f'Coefficient sign check: FAIL -- {len(sign_issues)} sign reversals')

In [ ]:
# Handle sign reversals if any
if not signs_consistent:
    print('Removing variables with sign reversals (positive coef when predicting default)...')
    iv_values = {
        'Account Balance': 0.6660, 'Payment Status of Previous Credit': 0.2918,
        'Duration of Credit (month)': 0.2798, 'Value Savings/Stocks': 0.1925,
        'Purpose': 0.1676, 'Credit Amount': 0.1493,
        'Most valuable available asset': 0.1126, 'Age (years)': 0.1013
    }
    for var in sorted(sign_issues, key=lambda x: iv_values.get(x, 0)):
        selected_variables.remove(var)
        excluded_from_shortlist.append(var)
        print(f'  Removed {var} (IV={iv_values.get(var, "?")})')
    
    X_woe = db_woe[selected_variables].copy()
    X = sm.add_constant(X_woe)
    logit_model = sm.Logit(y, X)
    logit_res = logit_model.fit(disp=0)
    signs_consistent = all(logit_res.params[v] < 0 for v in selected_variables)
    print(f'After removal -- signs consistent: {signs_consistent}')
    
    summary_df = pd.DataFrame({
        'variable': ['const'] + selected_variables,
        'coef': logit_res.params.values,
        'std_err': logit_res.bse.values,
        'z': logit_res.tvalues.values,
        'p_value': logit_res.pvalues.values,
        'ci_lower': logit_res.conf_int()[0].values,
        'ci_upper': logit_res.conf_int()[1].values
    })
    print(summary_df.to_string(index=False))
else:
    print('No sign reversals -- no remediation needed.')

In [ ]:
# Check 3: VIF
X_vif = db_woe[selected_variables].copy()
vif_data = pd.DataFrame({
    'variable': selected_variables,
    'vif': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
print('VIF values:')
print(vif_data.to_string(index=False))

vif_flag = bool(vif_data['vif'].max() > 5)
if vif_flag:
    print(f'WARNING: VIF > 5 for: {vif_data[vif_data["vif"] > 5]["variable"].tolist()}')
else:
    print('VIF check: PASS -- all VIF < 5')

In [ ]:
# Model discrimination metrics
y_pred = logit_res.predict(X)
model_auc = pdt.auc_model(y_pred, y)
model_gini = 2 * model_auc - 1

fpr, tpr, thresholds = roc_curve(y, y_pred)
model_ks = float(max(tpr - fpr))

print(f'AUC:  {model_auc:.4f}')
print(f'Gini: {model_gini:.4f}')
print(f'KS:   {model_ks:.4f}')

if model_auc < 0.60:
    print('AUC check: FAIL')
elif model_auc < 0.70:
    print('AUC check: WARN -- weak discrimination')
else:
    print('AUC check: PASS')

In [ ]:
# Check 4: Score distribution
scores = pdt.scaled_score(y_pred, score=600, odd=50, pdo=20)

score_stats = {
    'min': float(np.min(scores)),
    'max': float(np.max(scores)),
    'mean': float(np.mean(scores)),
    'std': float(np.std(scores)),
    'pct_below_400': float(np.mean(scores < 400) * 100),
    'pct_above_800': float(np.mean(scores > 800) * 100)
}
for k, v in score_stats.items():
    print(f'  {k}: {v:.2f}')

score_range_ok = (score_stats['min'] >= 300 and score_stats['max'] <= 900)
extremes_ok = (score_stats['pct_below_400'] <= 5 and score_stats['pct_above_800'] <= 5)
print(f'Score distribution check: {"PASS" if (score_range_ok and extremes_ok) else "WARN"}')

In [ ]:
# Check 5: Decile monotonicity
score_df = pd.DataFrame({'score': scores, 'default': y})
score_df['decile'] = pd.qcut(score_df['score'], 10, labels=False, duplicates='drop')
decile_stats = score_df.groupby('decile').agg(
    n=('default', 'count'),
    n_defaults=('default', 'sum'),
    default_rate=('default', 'mean'),
    avg_score=('score', 'mean'),
    min_score=('score', 'min'),
    max_score=('score', 'max')
).reset_index()
print('Decile analysis:')
print(decile_stats.to_string(index=False))

dr_values = decile_stats['default_rate'].values
reversals = sum(1 for i in range(len(dr_values)-1) if dr_values[i] < dr_values[i+1])
decile_monotonic = reversals <= 1
print(f'\nDecile monotonicity: {"PASS" if decile_monotonic else "FAIL"} (reversals: {reversals})')

In [ ]:
# Check 6: Cross-validation and bootstrap stability
cv_result = pdt.kfold_vld(logit_res, db_woe, target, selected_variables, k=10)
print('K-fold cross-validation summary:')
print(cv_result.summary)
print(f'CV iter columns: {cv_result.iter.columns.tolist()}')

boots_result = pdt.boots_vld(logit_res, db_woe, target, selected_variables, B=500)
print('\nBootstrap validation summary:')
print(boots_result.summary)
print(f'Boots iter columns: {boots_result.iter.columns.tolist()}')

In [ ]:
# Extract validation AUC
def get_auc_from_result(vld_result, fallback):
    try:
        cols = vld_result.iter.columns
        auc_cols = [c for c in cols if 'auc' in str(c).lower()]
        if auc_cols:
            return float(vld_result.iter[auc_cols[0]].mean())
        if hasattr(vld_result.summary, 'loc'):
            for idx in vld_result.summary.index:
                if 'auc' in str(idx).lower():
                    return float(vld_result.summary.loc[idx].iloc[0])
        return fallback
    except Exception:
        return fallback

cv_auc_mean = get_auc_from_result(cv_result, model_auc)
boots_auc_mean = get_auc_from_result(boots_result, model_auc)

cv_stable = abs(model_auc - cv_auc_mean) <= 0.03
boots_stable = abs(model_auc - boots_auc_mean) <= 0.03

print(f'Development AUC: {model_auc:.4f}')
print(f'CV AUC mean:     {cv_auc_mean:.4f} (diff: {abs(model_auc - cv_auc_mean):.4f}) -- {"PASS" if cv_stable else "WARN"}')
print(f'Bootstrap AUC:   {boots_auc_mean:.4f} (diff: {abs(model_auc - boots_auc_mean):.4f}) -- {"PASS" if boots_stable else "WARN"}')

## Plots

In [ ]:
# ROC Curve
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(fpr, tpr, color=BLUE, lw=2, label=f'ROC curve (AUC = {model_auc:.4f})')
ax.plot([0, 1], [0, 1], color=GREY, lw=1, linestyle='--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Stage 04c (Forward Stepwise) -- ROC Curve')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.savefig(f'{RUN_DIR}/figures/04c_roc_curve.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
# Score Distribution
fig, ax = plt.subplots(figsize=(10, 6))
defaults_s = scores[y == 1]
non_defaults_s = scores[y == 0]
ax.hist(non_defaults_s, bins=30, alpha=0.6, color=BLUE, label=f'Non-default (n={len(non_defaults_s)})', density=True)
ax.hist(defaults_s, bins=30, alpha=0.6, color=RED, label=f'Default (n={len(defaults_s)})', density=True)
ax.set_xlabel('Score')
ax.set_ylabel('Density')
ax.set_title('Stage 04c (Forward Stepwise) -- Score Distribution')
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig(f'{RUN_DIR}/figures/04c_score_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
# Score Decile Table
fig, ax1 = plt.subplots(figsize=(10, 6))
x_pos = list(range(len(decile_stats)))
ax1.bar(x_pos, decile_stats['n'].values, color=BLUE, alpha=0.5, label='Count')
ax2 = ax1.twinx()
ax2.plot(x_pos, decile_stats['default_rate'].values * 100, color=RED, marker='o', lw=2, label='Default Rate (%)')
ax1.set_xlabel('Score Decile (low to high score)')
ax1.set_ylabel('Count', color=BLUE)
ax2.set_ylabel('Default Rate (%)', color=RED)
ax1.set_title('Stage 04c (Forward Stepwise) -- Score Decile Analysis')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f'D{i+1}' for i in x_pos])
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
ax1.grid(True, alpha=0.3)
plt.savefig(f'{RUN_DIR}/figures/04c_score_decile_table.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
# Coefficient Plot
coef_df = summary_df[summary_df['variable'] != 'const'].copy()
coef_df = coef_df.sort_values('coef', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, max(4, len(coef_df) * 0.6)))
y_positions = list(range(len(coef_df)))
# For default prediction, negative coefficients are expected (shown in blue)
colors = [BLUE if c < 0 else RED for c in coef_df['coef'].values]
ax.barh(y_positions, coef_df['coef'].values, color=colors, height=0.6)
ax.errorbar(coef_df['coef'].values, y_positions,
            xerr=[coef_df['coef'].values - coef_df['ci_lower'].values,
                  coef_df['ci_upper'].values - coef_df['coef'].values],
            fmt='none', color='black', capsize=3)
ax.set_yticks(y_positions)
ax.set_yticklabels(coef_df['variable'].values)
ax.axvline(x=0, color=GREY, linestyle='--', lw=1)
ax.set_xlabel('Coefficient')
ax.set_title('Stage 04c (Forward Stepwise) -- Logistic Regression Coefficients')
ax.grid(True, alpha=0.3, axis='x')
plt.savefig(f'{RUN_DIR}/figures/04c_coefficient_plot.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
# Write model_params_fwd.json
coefficients_dict = {v: float(logit_res.params[v]) for v in selected_variables}
intercept_val = float(logit_res.params['const'])

model_params = {
    'selection_method': 'forward_stepwise',
    'selected_variables': selected_variables,
    'woe_mappings': {v: woe_mappings[v] for v in selected_variables if v in woe_mappings},
    'coefficients': coefficients_dict,
    'intercept': intercept_val,
    'score_params': {
        'base_score': 600,
        'base_odds': 50,
        'pdo': 20
    },
    'model_auc': round(model_auc, 4),
    'model_gini': round(model_gini, 4),
    'model_ks': round(model_ks, 4)
}

with open(f'{RUN_DIR}/pipeline/model_params_fwd.json', 'w') as f:
    json.dump(model_params, f, indent=2)

print(f'Model params written to {RUN_DIR}/pipeline/model_params_fwd.json')
print(json.dumps(model_params, indent=2))

In [ ]:
# Collect all flags
model_flags = []
if not signs_consistent:
    model_flags.append('Coefficient sign reversal detected and remediated')
if vif_flag:
    model_flags.append(f'VIF > 5 for: {vif_data[vif_data["vif"] > 5]["variable"].tolist()}')
if model_auc < 0.70:
    model_flags.append(f'AUC below 0.70: {model_auc:.4f}')
if not decile_monotonic:
    model_flags.append(f'Decile monotonicity failed ({reversals} reversals)')
if not cv_stable:
    model_flags.append(f'CV AUC unstable (diff={abs(model_auc - cv_auc_mean):.4f})')
if not boots_stable:
    model_flags.append(f'Bootstrap AUC unstable (diff={abs(model_auc - boots_auc_mean):.4f})')
if threshold_adjusted:
    model_flags.append(f'P-value threshold adjusted to {p_threshold_used}')

print(f'Model flags: {model_flags if model_flags else "None"}')

# Self-assessment per variable
vif_dict = dict(zip(vif_data['variable'], vif_data['vif']))
sa_coefficients = []
for var in selected_variables:
    coef_val = float(logit_res.params[var])
    sa_coefficients.append({
        'variable': var,
        'coefficient': round(coef_val, 4),
        'woe_direction_consistent': coef_val < 0,  # negative is correct for default prediction
        'vif': round(float(vif_dict.get(var, 0)), 2)
    })

for item in sa_coefficients:
    print(f"  {item['variable']}: coef={item['coefficient']}, consistent={item['woe_direction_consistent']}, VIF={item['vif']}")

## Stage Summary

| Item | Value | Status |
|---|---|---|
| Selection method | Forward Stepwise | -- |
| Variables selected | see output | PASS |
| AUC | see output | see output |
| Gini | see output | see output |
| KS | see output | see output |
| Coefficient signs | see output | see output |
| VIF max | see output | see output |
| Score range | see output | see output |
| Decile monotonicity | see output | see output |
| CV stability | see output | see output |

**Flags for human review:** see output above

**Recommended action for next stage:** Proceed to model comparison (Stage 04x) where this model will be compared against MIV and XGBoost selection methods.